# 5. Process EIA residential customers and population

Builds `outputs/eia_residential_customers.csv`: one row per EIA utility × state with its number of residential customers (households) and estimated population.

Customers are counted under the utility that owns the **wires and meter**, since that's the utility an MSA needs approval from:

| Included | Why |
|---|---|
| `Sales_Ult_Cust`, Part A (bundled service) | Utilities that both deliver and sell power |
| `Sales_Ult_Cust`, Part C (delivery only) | Wires utilities in restructured states (e.g. PG&E, ComEd delivery customers) |
| `Delivery_Companies` | Texas ERCOT wires utilities (Oncor, CenterPoint, AEP Texas, TNMP...), which aren't in `Sales_Ult_Cust` |
| `Short_Form` | Small utilities reporting only total customers; residential customers estimated (see below) |

| Left out | Why |
|---|---|
| Part B (energy only) and Part D (Texas retail providers) | Retail suppliers; the same households are already counted under the wires utility |
| Ownership "Behind the Meter" | Third-party rooftop solar owners (Sunrun, Tesla...) reporting lease/PPA customers; already counted under their utility |
| Utility 99999 "Adjustment" rows | EIA's state-level net corrections; can't be tied to a utility |

Population is estimated from Census households by county (see the population section below).

In [ ]:
eia_year = None

In [ ]:
import re
import unicodedata

import pandas as pd

from common import ACS_YEAR, EIA_CUSTOMERS_CSV, EIA_YEAR, RAW_CENSUS_DIR, RAW_EIA_DIR, get_logger

log = get_logger("process_eia")
eia_year = eia_year or EIA_YEAR
eia_dir = RAW_EIA_DIR / str(eia_year)

ADJUSTMENT_UTILITY = 99999
SMALL_UTILITY_MAX_CUSTOMERS = 10_000
MIN_COMPARABLE_UTILITIES = 5


def read_sales_sheet(path, sheet):
    """Read a sheet in the Sales_Ult_Cust layout (three header rows). Column
    11 is residential customers; the last column is total customers."""
    df = pd.read_excel(path, sheet_name=sheet, header=2)
    df = df.rename(columns={df.columns[11]: "residential_customers", df.columns[-1]: "total_customers",
                            "Utility Name": "utility_name", "State": "state", "Ownership": "ownership"})
    df = df[pd.to_numeric(df["Utility Number"], errors="coerce").notna()].copy()
    df["utility_id"] = df["Utility Number"].astype(int)
    for col in ("residential_customers", "total_customers"):
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    return df


sales = read_sales_sheet(eia_dir / f"Sales_Ult_Cust_{eia_year}.xlsx", "States")
delivery = read_sales_sheet(eia_dir / f"Delivery_Companies_{eia_year}.xlsx", "Delivery Companies")
short = pd.read_excel(eia_dir / f"Short_Form_{eia_year}.xlsx", sheet_name="861S").rename(columns={
    "Utility Number": "utility_id", "Utility Name": "utility_name", "State": "state", "Ownership": "ownership"})
short["total_customers"] = pd.to_numeric(short["Total Customers"], errors="coerce").fillna(0)

## Wires utilities

In [ ]:
utilities = sales[(sales["utility_id"] != ADJUSTMENT_UTILITY) & (sales["ownership"] != "Behind the Meter")]
wires = pd.concat([
    utilities[utilities["Part"] == "A"].assign(source="bundled"),
    utilities[utilities["Part"] == "C"].assign(source="delivery"),
    delivery.assign(source="delivery (Texas wires)"),
]).assign(residential_estimated=False)
wires.groupby("source")["residential_customers"].agg(["size", "sum"])

## Short-form utilities

These report only total customers. Residential customers = total customers × the residential share of comparable small long-form utilities (bundled, under 10,000 customers):

1. same state and ownership type (municipal, co-op...), if there are at least 5 such utilities;
2. otherwise, the same ownership type nationally (at least 5);
3. otherwise, all small utilities nationally.

The minimum group size matters: a share based on one or two utilities can be badly off (e.g. one commercial-only utility would give every small municipal in its state a residential share of 0).

In [ ]:
small = utilities[(utilities["Part"] == "A") & utilities["total_customers"].between(1, SMALL_UTILITY_MAX_CUSTOMERS)]


def residential_share(groups):
    """Residential share per group, only for groups with enough utilities."""
    g = small.groupby(groups).agg(n=("utility_id", "size"), res=("residential_customers", "sum"),
                                  tot=("total_customers", "sum"))
    g = g[g["n"] >= MIN_COMPARABLE_UTILITIES]
    return g["res"] / g["tot"]


by_state_ownership, by_ownership = residential_share(["state", "ownership"]), residential_share("ownership")
us_share = small["residential_customers"].sum() / small["total_customers"].sum()
short["residential_share"] = [by_state_ownership.get((s, o), by_ownership.get(o, us_share))
                              for s, o in zip(short["state"], short["ownership"])]
short["residential_customers"] = (short["total_customers"] * short["residential_share"]).round()
short = short.assign(source="short form (residential estimated)", residential_estimated=True)
print(f"{len(short):,} short-form utilities, {short['residential_customers'].sum():,.0f} estimated residential "
      f"customers (US small-utility residential share {us_share:.1%})")
short["residential_share"].describe().round(3)

## Combine wires and short-form utilities

In [ ]:
cols = ["state", "utility_id", "utility_name", "ownership", "source", "residential_estimated", "residential_customers"]
customers = (
    pd.concat([wires[cols], short[cols]])
    .groupby(["state", "utility_id"], as_index=False)
    .agg(utility_name=("utility_name", "first"), ownership=("ownership", "first"),
         source=("source", lambda s: " + ".join(sorted(set(s)))),
         residential_estimated=("residential_estimated", "any"),
         residential_customers=("residential_customers", "sum"))
)
customers["residential_customers"] = customers["residential_customers"].astype(int)
customers.insert(0, "eia_year", eia_year)
customers = customers.sort_values(["state", "utility_name"]).reset_index(drop=True)
print(f"{len(customers):,} utility x state rows, {customers['residential_customers'].sum():,} residential customers")

## Population

Population = residential customers × people per household in the utility's service area, then calibrated to Census state totals:

1. **People per household by county**: Census people living in households ÷ households (notebook 3).
2. **People per household by utility**: over the counties EIA lists in its service territory, total people in households ÷ total households (a household-weighted average). EIA lists which counties a utility serves but not how much of each, so a utility serving part of a large county gets that whole county's average. Utilities with no matched county use their state's average.
3. **Raw estimate**: residential customers × that utility's people per household.
4. **Calibrate**: scale each state's utilities so they add up to the state's Census population in households. Residential customers outnumber households (vacation homes, vacant units, extra meters count as customers), so without this step population would be overstated, most in states with many seasonal homes.

In [ ]:
census = pd.read_csv(RAW_CENSUS_DIR / str(ACS_YEAR) / "county_households.csv")
territory = (pd.read_excel(eia_dir / f"Service_Territory_{eia_year}.xlsx", sheet_name="Counties_States")
             .rename(columns={"Utility Number": "utility_id", "State": "state", "County": "county"}))


def county_key(name):
    """Comparable county name: ASCII, lowercase, without 'County', 'Parish' etc.
    and without punctuation or spaces, so "Prince George's County" (Census)
    matches "Prince Georges" (EIA) and "De Soto" matches "DeSoto". Independent
    cities keep "city" ("Baltimore City" vs "Baltimore" county)."""
    s = unicodedata.normalize("NFKD", str(name).split(",")[0]).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"\b(city and borough|census area|county|parish|borough|municipality|planning region)\b", "", s)
    s = re.sub(r"\bsaint\b", "st", s)
    return re.sub(r"[^a-z0-9]", "", s)


census["key"] = census["county_name"].map(county_key)
assert not census.duplicated(["state", "key"]).any(), "two Census counties share a key"
territory["key"] = territory["county"].map(county_key)
territory = (territory.drop_duplicates(["state", "utility_id", "key"])
             .merge(census[["state", "key", "households", "household_population"]], on=["state", "key"], how="left"))

unmatched = territory[territory["households"].isna()]
log.info("population  EIA service-territory counties matched to Census: %d of %d; unmatched by state: %s",
         len(territory) - len(unmatched), len(territory),
         ", ".join(f"{s} {n}" for s, n in unmatched.groupby("state").size().sort_values(ascending=False).items()))
unmatched[["state", "county"]].drop_duplicates().groupby("state")["county"].apply(list)

In [ ]:
by_utility = territory.dropna(subset=["households"]).groupby(["state", "utility_id"])[
    ["households", "household_population"]].sum()
by_utility["people_per_household"] = by_utility["household_population"] / by_utility["households"]
by_state = census.groupby("state")[["households", "household_population"]].sum()
state_people_per_household = by_state["household_population"] / by_state["households"]

customers = customers.drop(columns=["people_per_household", "people_per_household_source", "population"],
                           errors="ignore").merge(
    by_utility["people_per_household"].reset_index(), on=["state", "utility_id"], how="left")
customers["people_per_household_source"] = customers["people_per_household"].notna().map(
    {True: "service area counties", False: "state average"})
customers["people_per_household"] = customers["people_per_household"].fillna(customers["state"].map(state_people_per_household))

raw = customers["residential_customers"] * customers["people_per_household"]
calibration = by_state["household_population"] / raw.groupby(customers["state"]).sum()
customers["population"] = (raw * customers["state"].map(calibration)).round().astype(int)
customers["people_per_household"] = customers["people_per_household"].round(3)

log.info("population  %s people in households; state calibration factors %.2f-%.2f (median %.2f); "
         "%d utilities use their state's average people per household",
         f"{customers['population'].sum():,}", calibration.min(), calibration.max(), calibration.median(),
         (customers["people_per_household_source"] == "state average").sum())
calibration.rename("calibration_factor").sort_values().round(3).to_frame().T

## Write `eia_residential_customers.csv`

In [ ]:
customers.to_csv(EIA_CUSTOMERS_CSV, index=False)
log.info("wrote %s: %d utility x state rows, %s residential customers (%s estimated for short-form utilities), "
         "%s people", EIA_CUSTOMERS_CSV.name, len(customers), f"{customers['residential_customers'].sum():,}",
         f"{customers.loc[customers['residential_estimated'], 'residential_customers'].sum():,}",
         f"{customers['population'].sum():,}")
customers.sort_values("population", ascending=False).head(10)[
    ["state", "utility_name", "residential_customers", "people_per_household", "population"]]

## Check against EIA's published state totals

EIA's published residential customer totals count each customer once on the seller side (Parts A, B, D, including the adjustment rows). Ours should be close; Texas runs a little high because the wires utilities report slightly more customers than the retail providers.

In [ ]:
published = sales[sales["Part"].isin(["A", "B", "D"])].groupby("state")["residential_customers"].sum()
check = pd.DataFrame({"ours": customers.groupby("state")["residential_customers"].sum(), "eia_published": published})
check["pct_diff"] = (check["ours"] / check["eia_published"] - 1) * 100
us_diff = (check["ours"].sum() / check["eia_published"].sum() - 1) * 100
log.info("check  US total %+.2f%% vs EIA published; states off by >1%%: %s", us_diff,
         ", ".join(f"{s} {d:+.1f}%" for s, d in check["pct_diff"].items() if abs(d) > 1) or "none")
check[check["pct_diff"].abs() > 1].round(1)